In [ ]:
# -*- coding: utf-8 -*-
"""
Módulo Optimizado de Clasificación Automatizada de Estilos Musicales mediante
Segmentación en Memoria, Cuantización Vectorial Masiva y CNN 1D.

Optimizaciones: Vectorización completa mediante NumPy y supresión de operaciones I/O redundantes.
"""

import os
import random
import kagglehub
import pandas as pd
import numpy as np
import tensorflow as tf
from google.colab import drive
from sklearn.cluster import MiniBatchKMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, Embedding, Conv1D, MaxPooling1D, GlobalMaxPooling1D
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import regularizers
from tensorflow.keras.preprocessing.sequence import pad_sequences
import warnings

warnings.filterwarnings("ignore", category=FutureWarning, module="librosa")
warnings.filterwarnings("ignore", category=UserWarning, module="librosa")

In [ ]:
# =====================================================================
# 1. CONEXIÓN AL ENTORNO DE ALMACENIMIENTO
# =====================================================================
print("--- Etapa 1: Conexión a Repositorios Persistentes ---")
drive.mount('/content/drive')

# Rutas de origen (Donde ya residen las características pre-calculadas)
ruta_salida_base = '/content/drive/MyDrive/Colab Notebooks/Kaggle_files/Train/'
carpeta_mfccs = os.path.join(ruta_salida_base, 'mfccs/') # Carpeta con los .npy originales de 13xVentanas


In [ ]:
# =====================================================================
# 2. CARGA EFICIENTE Y SEGMENTACIÓN EN MEMORIA (IN-MEMORY CHUNKING)
# =====================================================================
print("\n--- Etapa 2: Segmentación y Alineación Vectorial en Memoria RAM ---")
df_train = pd.read_csv('train.csv')

X_mfcc_chunks_list = []
y_chunk_labels_list = []
cancion_mapeo_indices = {} # Estructura para rastrear qué chunks pertenecen a qué canción (Útil para Test)

# Parámetros equivalentes de segmentación temporal calculados sobre marcos (frames) de MFCC
# Asumiendo una tasa de muestreo estándar donde 10 segundos equivalen aproximadamente a 430 frames
CHUNK_FRAMES = 430
HOP_FRAMES = int(CHUNK_FRAMES * 0.5) # 50% de solapamiento

chunk_counter = 0

for index, row in df_train.iterrows():
    nombre_base = row['filename']
    id_cancion = os.path.splitext(nombre_base)[0]
    etiqueta = int(row['label'])

    nombre_archivo_mfcc = f"{id_cancion}_mfccs_entrenamiento.npy"
    ruta_completa_mfcc = os.path.join(carpeta_mfccs, nombre_archivo_mfcc)

    if not os.path.exists(ruta_completa_mfcc):
        continue

    # Carga de la matriz pre-calculada. Forma esperada: (13, frames_totales)
    mfcc_cancion = np.load(ruta_completa_mfcc)

    # Transposición matemática inmediata: (frames_totales, 13)
    mfcc_cancion = mfcc_cancion.T
    frames_totales = mfcc_cancion.shape[0]

    chunks_de_esta_cancion = []

    # Segmentación por ventanas deslizantes directamente sobre la matriz en memoria RAM
    for start in range(0, frames_totales - CHUNK_FRAMES + 1, HOP_FRAMES):
        chunk = mfcc_cancion[start:start + CHUNK_FRAMES, :]
        X_mfcc_chunks_list.append(chunk)
        y_chunk_labels_list.append(etiqueta)

        chunks_de_esta_cancion.append(chunk_counter)
        chunk_counter += 1

    if chunks_de_esta_cancion:
        cancion_mapeo_indices[nombre_base] = chunks_de_esta_cancion

print(f"Total de fragmentos acústicos estructurados en memoria: {len(X_mfcc_chunks_list)}")


In [ ]:
# =====================================================================
# 3. ENTRENAMIENTO EXPRESO DEL CODEBOOK GLOBAL
# =====================================================================
print("\n--- Etapa 3: Cuantización Vectorial Masiva (Batch Quantization) ---")

# Indexación aleatoria de fragmentos en memoria para el ajuste del clasificador no supervisado
np.random.seed(42)
indices_muestra = np.random.choice(len(X_mfcc_chunks_list), size=min(600, len(X_mfcc_chunks_list)), replace=False)
muestra_para_fit = [X_mfcc_chunks_list[idx] for idx in indices_muestra]

X_codebook_train = np.vstack(muestra_para_fit)

kmeans_global = MiniBatchKMeans(n_clusters=100, random_state=42, batch_size=8192) # Ajuste de batch para optimizar GPU/CPU
kmeans_global.fit(X_codebook_train)
print("Estado: Codebook global consolidado de forma determinista.")

In [ ]:
# =====================================================================
# 4. TOKENIZACIÓN VECTORIAL EN BLOQUE Y PREPARACIÓN DE TENSORES
# =====================================================================
# En lugar de iterar por archivos, concatenamos longitudinalmente todos los chunks en un arreglo único
X_all_chunks_stacked = np.vstack(X_mfcc_chunks_list)

# Computación simultánea de todas las asignaciones de centroides (Operación masiva de alta velocidad)
all_tokens_predicted = kmeans_global.predict(X_all_chunks_stacked)

# Re-estructuramos el vector plano resultante de tokens de vuelta a sus respectivas dimensiones de secuencias
X_tokens_chunks_list = np.split(all_tokens_predicted, len(X_mfcc_chunks_list))

# Homogeneización dimensional mediante la adición de padding pos-secuencia
MAX_LEN = 300
X_final = pad_sequences(X_tokens_chunks_list, maxlen=MAX_LEN, padding='post', truncating='post')
y_final = np.array(y_chunk_labels_list)

# División tripartita estratificada del dataset optimizado
X_temp, X_test, y_temp, y_test = train_test_split(
    X_final, y_final, test_size=0.1, stratify=y_final, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.2/0.9, stratify=y_temp, random_state=42
)

print(f"Estructuras del diseño experimental finalizadas -> Train: {X_train.shape}, Validation: {X_val.shape}, Test: {X_test.shape}")


In [ ]:
# =====================================================================
# 5. CONSTRUCCIÓN Y EVALUACIÓN DE TOPOLOGÍA CONVOLUCIONAL 1D
# =====================================================================
def calculate_f1(y_true, y_pred_probs):
    y_pred = np.argmax(y_pred_probs, axis=1)
    return f1_score(y_true, y_pred, average='macro')

def train_evaluate_cnn1d(activation, depth, neurons, learning_rate, optimizer_name='Adam',
                         batch_size=32, initializer='glorot_uniform', dropout_rate=0.2,
                         regularizer_type='l2', regularizer_lambda=0.0001, epochs=150, patience=25,
                         vocab_size=100, max_length=300):

    reg = regularizers.l2(regularizer_lambda) if regularizer_type == 'l2' else None

    model = Sequential()
    model.add(Input(shape=(max_length,), dtype='int32'))
    model.add(Embedding(input_dim=vocab_size, output_dim=64))

    model.add(Conv1D(filters=neurons, kernel_size=7, activation=activation, kernel_initializer=initializer, kernel_regularizer=reg, padding='same'))
    model.add(MaxPooling1D(pool_size=3))
    if dropout_rate > 0: model.add(Dropout(dropout_rate))

    for _ in range(depth - 1):
        model.add(Conv1D(filters=neurons, kernel_size=5, activation=activation, kernel_initializer=initializer, kernel_regularizer=reg, padding='same'))
        model.add(MaxPooling1D(pool_size=2))
        if dropout_rate > 0: model.add(Dropout(dropout_rate))

    model.add(GlobalMaxPooling1D())

    model.add(Dense(neurons, activation=activation, kernel_initializer=initializer, kernel_regularizer=reg))
    if dropout_rate > 0: model.add(Dropout(dropout_rate))
    model.add(Dense(8, activation='softmax'))

    if optimizer_name == 'Adam': optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer_name == 'RMSprop': optimizer = tf.keras.optimizers.RMSprop(learning_rate=learning_rate)
    else: optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate)

    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    early_stop = EarlyStopping(monitor='val_loss', patience=payout_rate if 'payout_rate' in locals() else patience, restore_best_weights=True)

    history = model.fit(
        X_train, y_train, validation_data=(X_val, y_val),
        epochs=epochs, batch_size=batch_size, callbacks=[early_stop], verbose=0
    )

    y_val_pred_probs = model.predict(X_val, verbose=0)
    y_test_pred_probs = model.predict(X_test, verbose=0)

    return history, calculate_f1(y_val, y_val_pred_probs), calculate_f1(y_test, y_test_pred_probs), model

In [ ]:
# =====================================================================
# 6. OPTIMIZACIÓN DE HIPERPARÁMETROS (RANDOM SEARCH)
# =====================================================================
print("\n--- Etapa 4: Ejecución del algoritmo de búsqueda aleatoria ---")
activations_list  = ['relu', 'tanh']
depths            = [1, 2, 3]
neurons_list      = [32, 64, 128]
learning_rates    = [0.001, 0.01]

fixed_batch_size  = 32
fixed_initializer = 'glorot_uniform'
fixed_optimizer   = 'Adam'
fixed_patience    = 20

num_trials = 20
results = []
best_f1 = -1
best_config = None

for i in range(num_trials):
    activation = random.choice(activations_list)
    depth = random.choice(depths)
    neurons = random.choice(neurons_list)
    lr = random.choice(learning_rates)

    print(f"\nEvaluación de Trial {i+1}/{num_trials} -> Topología CNN: act={activation}, depth={depth}, filtros={neurons}, lr={lr}")

    history, val_f1, test_f1, model = train_evaluate_cnn1d(
        activation=activation, depth=depth, neurons=neurons, learning_rate=lr,
        optimizer_name=fixed_optimizer, batch_size=fixed_batch_size,
        initializer=fixed_initializer, epochs=150, patience=fixed_patience
    )

    results.append({
        'activation': activation, 'depth': depth, 'neurons': neurons, 'learning_rate': lr, 'val_f1': val_f1, 'test_f1': test_f1
    })
    print(f"Métricas del Trial -> Validación F1: {val_f1:.4f} | Test F1: {test_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_config = results[-1]
        best_model_keras = model
        print("Notificación: Actualización del mejor estimador en memoria.")

print("\n===== ARQUITECTURA ÓPTIMA CONSOLIDADA =====")
print(best_config)

In [ ]:
# =====================================================================
# 7. INFERENCIA MEDIANTE ENSEMBLE POR VOTACIÓN (MÓDULO DE PRUEBA KAGGLE)
# =====================================================================
print("\n--- Etapa 5: Inferencia por ensamble de fragmentos (Votación) ---")
df_test = pd.read_csv('test.csv')
carpeta_mfccs_test = '/content/drive/MyDrive/Colab Notebooks/Kaggle_files/Test/mfccs/'

estilos_dict = np.load('dict.npy', allow_pickle=True).item()
inverso_estilos_dict = {v: k for k, v in estilos_dict.items()}

nombres_archivos_salida = []
predicciones_finales_texto = []

for index, row in df_test.iterrows():
    nombre_base = row['filename']
    id_cancion = os.path.splitext(nombre_base)[0]

    nombre_archivo_mfcc = f"{id_cancion}_chunks.npy" # Ajustar a nomenclatura de guardado de Test
    ruta_completa_mfcc = os.path.join(carpeta_mfccs_test, nombre_archivo_mfcc)

    if os.path.exists(ruta_completa_mfcc):
        mfcc_cancion_test = np.load(ruta_completa_mfcc).T
        frames_totales_test = mfcc_cancion_test.shape[0]

        chunks_tokens_cancion = []

        # Segmentación homóloga en memoria RAM para el set de validación externa
        for start in range(0, frames_totales_test - CHUNK_FRAMES + 1, HOP_FRAMES):
            chunk_test = mfcc_cancion_test[start:start + CHUNK_FRAMES, :]
            # Predicción con el cuantizador estático universal
            tokens_test_chunk = kmeans_global.predict(chunk_test)
            chunks_tokens_cancion.append(tokens_test_chunk)

        if chunks_tokens_cancion:
            X_chunks_pred = pad_sequences(chunks_tokens_cancion, maxlen=MAX_LEN, padding='post', truncating='post')
            probabilidades_chunks = best_model_keras.predict(X_chunks_pred, verbose=0)
            clases_chunks = np.argmax(probabilidades_chunks, axis=1)

            # Criterio de agregación estadística mediante la moda empírica de votos
            clase_ganadora = np.bincount(clases_chunks).argmax()

            nombres_archivos_salida.append(nombre_base)
            predicciones_finales_texto.append(inverso_estilos_dict[clase_ganadora])

# Consolidación del DataFrame final y exportación a almacenamiento local
submission = pd.DataFrame({'filename': nombres_archivos_salida, 'label': predicciones_finales_texto})
submission.to_csv('submission_kaggle_mir.csv', index=False)

print("\nPipeline completado de forma canónica. Archivo exportado para evaluación externa.")
print(submission.head(10))